# Dataset Information for Restaurant ABSA

This notebook summarizes the dataset used in the project **Aspect-Based Sentiment Analysis for Restaurant Reviews**.

## 1. Dataset Source

This project uses a standard public dataset from **SemEval-2014 Task 4: Aspect Based Sentiment Analysis**.

The selected subset is the **restaurant review dataset**, which contains English restaurant review sentences with human-annotated aspect categories and sentiment polarities.

In this project, the raw file is stored at:

`backends/data/raw/Restaurants_Train_v2.xml`

The processed files are stored at:

- `backends/data/processed/restaurant_absa_4_aspects.csv`
- `backends/data/processed/restaurant_absa_aspect_rows.csv`

## 2. Why This Dataset Is Suitable

The project topic is Aspect-Based Sentiment Analysis for restaurant reviews. This dataset is suitable because:

- It is already in the **restaurant domain**.
- It contains **real review sentences** written in English.
- It provides **aspect-level annotations**, not only full-sentence sentiment labels.
- It includes restaurant-related categories such as food, service, price, and ambience.
- It can be converted into the exact project output format: Food, Service, Price, and Eating Environment / Ambiance.

In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET
from collections import Counter

import pandas as pd

DATA_DIR = Path.cwd()
RAW_XML_PATH = DATA_DIR / "raw" / "Restaurants_Train_v2.xml"
WIDE_CSV_PATH = DATA_DIR / "processed" / "restaurant_absa_4_aspects.csv"
LONG_CSV_PATH = DATA_DIR / "processed" / "restaurant_absa_aspect_rows.csv"

print("Data folder:", DATA_DIR)
print("Raw XML exists:", RAW_XML_PATH.exists(), RAW_XML_PATH)
print("Wide CSV exists:", WIDE_CSV_PATH.exists(), WIDE_CSV_PATH)
print("Long CSV exists:", LONG_CSV_PATH.exists(), LONG_CSV_PATH)

Data folder: /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data
Raw XML exists: True /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/raw/Restaurants_Train_v2.xml
Wide CSV exists: True /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/processed/restaurant_absa_4_aspects.csv
Long CSV exists: True /Users/kafe/Desktop/NLP_Final Project/absa_v01/backends/data/processed/restaurant_absa_aspect_rows.csv


## 3. Raw Dataset Content

The original XML file contains review sentences. Each sentence may have aspect category annotations and polarity annotations.

The original aspect categories include:

- `food`
- `service`
- `price`
- `ambience`
- `anecdotes/miscellaneous`

The original polarity labels include:

- `positive`
- `negative`
- `neutral`
- `conflict`

In [2]:
tree = ET.parse(RAW_XML_PATH)
root = tree.getroot()

raw_category_counts = Counter()
raw_polarity_counts = Counter()
raw_sentence_count = 0

for sentence in root.findall("sentence"):
    raw_sentence_count += 1
    for category_node in sentence.findall("./aspectCategories/aspectCategory"):
        raw_category_counts[category_node.attrib.get("category", "Unknown")] += 1
        raw_polarity_counts[category_node.attrib.get("polarity", "Unknown")] += 1

print("Number of raw review sentences:", raw_sentence_count)

raw_category_df = pd.DataFrame(
    raw_category_counts.items(), columns=["Original Aspect Category", "Count"]
).sort_values("Count", ascending=False)

raw_polarity_df = pd.DataFrame(
    raw_polarity_counts.items(), columns=["Original Polarity", "Count"]
).sort_values("Count", ascending=False)

display(raw_category_df)
display(raw_polarity_df)

Number of raw review sentences: 3041


,Original Aspect Category,Count
1,food,1232
2,anecdotes/miscellaneous,1132
0,service,597
4,ambience,431
3,price,321


,Original Polarity,Count
1,positive,2179
0,negative,839
3,neutral,500
2,conflict,195


## 4. Project Label Mapping

The project uses four fixed aspects. The original SemEval categories are mapped into the project aspects as follows:

In [3]:
mapping_df = pd.DataFrame(
    [
        ("food", "Food", "Used"),
        ("service", "Service", "Used"),
        ("price", "Price", "Used"),
        ("ambience", "Eating Environment / Ambiance", "Used"),
        ("anecdotes/miscellaneous", "None", "Ignored because it is outside the four project aspects"),
    ],
    columns=["Original Category", "Project Aspect", "Decision"],
)

display(mapping_df)

,Original Category,Project Aspect,Decision
0,food,Food,Used
1,service,Service,Used
2,price,Price,Used
3,ambience,Eating Environment / Ambiance,Used
4,anecdotes/miscellaneous,None,Ignored because it is outside the four project...


The final project sentiment labels are:

- **Positive**: the review expresses a good opinion about the aspect.
- **Negative**: the review expresses a bad opinion about the aspect.
- **Unknown**: the aspect is not mentioned, unclear, outside the project scope, or cannot be confidently classified.

The `Unknown` label is important because each review does not always mention all four aspects.

## 5. Processed Dataset Files

The preprocessing notebook creates two processed files:

1. **Wide format**: `restaurant_absa_4_aspects.csv`
   - One row per review.
   - Four aspect label columns.
   - Useful for traditional ML and checking full review output.

2. **Long format**: `restaurant_absa_aspect_rows.csv`
   - One row per review-aspect pair.
   - Each review creates four rows.
   - Useful for BERT/deep learning because the model receives `(review, aspect)` as input.

In [4]:
wide_df = pd.read_csv(WIDE_CSV_PATH, dtype={"id": str})
long_df = pd.read_csv(LONG_CSV_PATH, dtype={"id": str})

print("Wide dataset shape:", wide_df.shape)
print("Long dataset shape:", long_df.shape)
print("Number of unique reviews:", wide_df["id"].nunique())
print("Expected long rows = reviews x 4 aspects:", len(wide_df) * 4)

display(wide_df.head())
display(long_df.head(8))

Wide dataset shape: (3041, 6)
Long dataset shape: (12164, 4)
Number of unique reviews: 3041
Expected long rows = reviews x 4 aspects: 12164


,id,review,Food,Service,Price,Eating Environment / Ambiance
0,3121,But the staff was so horrible to us.,Unknown,Negative,Unknown,Unknown
1,2777,"To be completely fair, the only redeeming fact...",Positive,Unknown,Unknown,Unknown
2,1634,"The food is uniformly exceptional, with a very...",Positive,Unknown,Unknown,Unknown
3,2534,Where Gabriela personaly greets you and recomm...,Unknown,Positive,Unknown,Unknown
4,583,"For those that go once and don't enjoy it, all...",Unknown,Unknown,Unknown,Unknown


,id,review,aspect,sentiment
0,3121,But the staff was so horrible to us.,Food,Unknown
1,3121,But the staff was so horrible to us.,Service,Negative
2,3121,But the staff was so horrible to us.,Price,Unknown
3,3121,But the staff was so horrible to us.,Eating Environment / Ambiance,Unknown
4,2777,"To be completely fair, the only redeeming fact...",Food,Positive
5,2777,"To be completely fair, the only redeeming fact...",Service,Unknown
6,2777,"To be completely fair, the only redeeming fact...",Price,Unknown
7,2777,"To be completely fair, the only redeeming fact...",Eating Environment / Ambiance,Unknown


## 6. Processed Label Distribution

The following table shows how many Positive, Negative, and Unknown labels exist for each project aspect.

In [5]:
PROJECT_ASPECTS = [
    "Food",
    "Service",
    "Price",
    "Eating Environment / Ambiance",
]
LABEL_ORDER = ["Positive", "Negative", "Unknown"]

label_distribution = pd.DataFrame(
    {
        aspect: wide_df[aspect].value_counts().reindex(LABEL_ORDER, fill_value=0)
        for aspect in PROJECT_ASPECTS
    }
).T

label_distribution.index.name = "Aspect"
display(label_distribution)

overall_label_distribution = wide_df[PROJECT_ASPECTS].stack().value_counts().reindex(LABEL_ORDER, fill_value=0)
display(overall_label_distribution.to_frame(name="Total Count"))

,Positive,Negative,Unknown
Aspect,,,
Food,867,209,1965
Service,324,218,2499
Price,179,115,2747
Eating Environment / Ambiance,263,98,2680


,Total Count
Positive,1633
Negative,640
Unknown,9891


## 7. Dataset Size and Imbalance

The dataset is large enough for this course project because it contains thousands of real restaurant review sentences and more than twelve thousand review-aspect samples after preprocessing.

However, the dataset is imbalanced. Many labels are `Unknown` because most restaurant reviews do not discuss all four aspects at the same time. This is why the project uses **F1-score**, especially macro F1-score, in addition to accuracy. Accuracy alone may be misleading if a model predicts `Unknown` too often.

In [6]:
total_aspect_samples = len(long_df)
overall_percentages = (overall_label_distribution / total_aspect_samples * 100).round(2)

summary_df = pd.DataFrame(
    {
        "Count": overall_label_distribution,
        "Percentage": overall_percentages.astype(str) + "%",
    }
)

print("Total review-level rows:", len(wide_df))
print("Total aspect-level rows:", total_aspect_samples)
display(summary_df)

Total review-level rows: 3041
Total aspect-level rows: 12164


,Count,Percentage
Positive,1633,13.42%
Negative,640,5.26%
Unknown,9891,81.31%


## 8. How the System Uses This Dataset

The system is trained and tested on the processed dataset:

- The **traditional machine learning method** uses the wide CSV file and trains one classifier for each aspect.
- The **deep learning method** uses the long review-aspect pair CSV file and predicts the sentiment for each `(review, aspect)` pair.
- The system output always contains all four aspects.
- Evaluation uses Accuracy, Precision, Recall, and F1-score to measure model quality.

This satisfies the dataset requirement because the dataset is introduced, comes from a standard public ABSA benchmark, matches the restaurant ABSA problem, contains clear aspect and sentiment labels, and is used for model training and evaluation.